# 📗 Bài 3 — Xác suất & thống kê

**Tuần 5** · Đủ dùng cho công việc, không phải để thi.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (11, 4)
rng = np.random.default_rng(42)

## 1. Mean vs Median — ví dụ lương

In [ ]:
luong = np.array([10, 10, 10, 10, 10, 10, 10, 10, 10, 1000])  # triệu VND

print(f"  Mean   : {luong.mean():8.1f} triệu")
print(f"  Median : {np.median(luong):8.1f} triệu")
print(f"  Std    : {luong.std():8.1f}")
print()
print("  'Lương trung bình công ty 109 triệu' — đúng về số học, sai về ý nghĩa.")

In [ ]:
plt.hist(luong, bins=30, edgecolor="black", alpha=.7)
plt.axvline(luong.mean(), color="red", ls="--", lw=2, label=f"mean = {luong.mean():.0f}")
plt.axvline(np.median(luong), color="green", ls="--", lw=2, label=f"median = {np.median(luong):.0f}")
plt.xlabel("lương (triệu)"); plt.legend(); plt.title("Một giá trị cực đoan kéo mean đi rất xa")
plt.show()

| Chỉ số | Nhạy ngoại lai | Dùng khi |
|---|---|---|
| **Mean** | ✅ Rất nhạy | Phân bố đối xứng, không ngoại lai |
| **Median** | ❌ Bền vững | Có ngoại lai hoặc phân bố lệch |
| **Mode** | ❌ | Dữ liệu hạng mục |

> 💡 **Dấu hiệu chẩn đoán:** mean và median lệch nhau nhiều → phân bố lệch hoặc có ngoại lai. Hãy vẽ histogram trước khi làm gì tiếp.
>
> Cũng vì thế, khi **điền giá trị thiếu** thì median an toàn hơn mean.

## 2. Độ lệch chuẩn

In [ ]:
hep = rng.normal(50, 5, 1000)
rong = rng.normal(50, 20, 1000)

fig, (a1, a2) = plt.subplots(1, 2, sharey=True)
a1.hist(hep, bins=40, alpha=.7, color="steelblue"); a1.set_title(f"std nhỏ = {hep.std():.1f}")
a2.hist(rong, bins=40, alpha=.7, color="coral");    a2.set_title(f"std lớn = {rong.std():.1f}")
for a in (a1, a2):
    a.axvline(50, color="red", ls="--"); a.set_xlim(-20, 120)
plt.tight_layout(); plt.show()

print(f"Cùng mean ({hep.mean():.1f} vs {rong.mean():.1f}) nhưng phân tán rất khác nhau.")

## 3. Phân phối chuẩn & quy tắc 68–95–99.7

In [ ]:
x = rng.normal(0, 1, 100_000)

for k in (1, 2, 3):
    ty_le = np.mean(np.abs(x) < k) * 100
    print(f"  Trong ±{k} std: {ty_le:5.2f}%")

plt.hist(x, bins=100, density=True, alpha=.6)
for k, mau in [(1, "green"), (2, "orange"), (3, "red")]:
    plt.axvline(k, color=mau, ls="--"); plt.axvline(-k, color=mau, ls="--")
plt.title("Phân phối chuẩn — quy tắc 68–95–99.7"); plt.show()

> Đây là cơ sở của phát hiện ngoại lai bằng **z-score**: điểm cách mean quá 3 std chỉ chiếm ~0.3% — đáng nghi ngờ.

## 4. Phát hiện ngoại lai

In [ ]:
du_lieu = np.concatenate([rng.normal(100, 15, 200), [400, 450, -50]])   # 3 ngoại lai

# Cách 1: z-score
z = np.abs((du_lieu - du_lieu.mean()) / du_lieu.std())
print("Z-score  > 3 :", du_lieu[z > 3].round(1))

# Cách 2: IQR (bền vững hơn — không bị chính ngoại lai làm lệch)
q1, q3 = np.percentile(du_lieu, [25, 75])
iqr = q3 - q1
duoi, tren = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print("IQR      ngoài [%.1f, %.1f]: %s" % (duoi, tren, du_lieu[(du_lieu < duoi) | (du_lieu > tren)].round(1)))

> **IQR thường tốt hơn z-score** vì mean và std bản thân chúng đã bị ngoại lai kéo lệch. Quartile thì không. Đây là phương pháp `boxplot` dùng — bạn sẽ gặp lại ở Tuần 7.

## 5. Tương quan

In [ ]:
n = 200
x = rng.normal(0, 1, n)

bo_du_lieu = [
    ("r ≈ +1  (cùng chiều)", x, x * 2 + rng.normal(0, .3, n)),
    ("r ≈  0  (không liên quan)", x, rng.normal(0, 1, n)),
    ("r ≈ -1  (ngược chiều)", x, -x * 2 + rng.normal(0, .3, n)),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (ten, a, b) in zip(axes, bo_du_lieu):
    r = np.corrcoef(a, b)[0, 1]
    ax.scatter(a, b, alpha=.5, s=15)
    ax.set_title(f"{ten}\nr = {r:.3f}")
    ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

### ⚠️ Tương quan bỏ sót quan hệ phi tuyến

In [ ]:
x = np.linspace(-3, 3, 200)
y = x**2 + rng.normal(0, .5, 200)

plt.scatter(x, y, alpha=.5, s=15)
plt.title(f"Quan hệ RẤT rõ ràng, nhưng r = {np.corrcoef(x, y)[0,1]:.3f}")
plt.grid(alpha=.3); plt.show()

print("Hệ số tương quan chỉ đo quan hệ TUYẾN TÍNH.")
print("=> LUÔN vẽ biểu đồ, đừng chỉ nhìn con số.")

## 6. ⚠️ Tương quan ≠ nhân quả

In [ ]:
# Biến ẩn: nhiệt độ
nhiet_do = rng.uniform(20, 40, 300)

kem = nhiet_do * 50 + rng.normal(0, 100, 300)          # nóng -> ăn kem nhiều
duoi_nuoc = nhiet_do * 0.3 + rng.normal(0, 1, 300)     # nóng -> đi bơi nhiều

r = np.corrcoef(kem, duoi_nuoc)[0, 1]
plt.scatter(kem, duoi_nuoc, alpha=.5, s=15)
plt.xlabel("doanh số kem"); plt.ylabel("số vụ đuối nước")
plt.title(f"r = {r:.3f} — nhưng kem KHÔNG gây đuối nước")
plt.grid(alpha=.3); plt.show()

**Bốn khả năng khi thấy A tương quan với B:**
1. A gây ra B
2. B gây ra A
3. C gây ra cả A và B (*confounder* — ở đây là **trời nóng**)
4. Trùng hợp ngẫu nhiên trên dữ liệu nhỏ

> ⚠️ **Vì sao AI Engineer phải nhớ:** model chỉ học tương quan. Model bảo *"khách gọi tổng đài nhiều → sắp rời bỏ"* **không** có nghĩa là chặn tổng đài sẽ giữ được khách.
>
> Cách duy nhất chứng minh nhân quả là **can thiệp có kiểm soát** — A/B test.

## 7. Xác suất — nền tảng của phân loại

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z = np.linspace(-8, 8, 200)
plt.plot(z, sigmoid(z), lw=2)
plt.axhline(.5, color="red", ls="--", alpha=.6)
plt.axvline(0, color="gray", ls=":", alpha=.6)
plt.xlabel("z (đầu ra tuyến tính)"); plt.ylabel("xác suất")
plt.title("Sigmoid biến số thực bất kỳ thành xác suất (0, 1)")
plt.grid(alpha=.3); plt.show()

for v in (-5, -2, 0, 2, 5):
    print(f"  sigmoid({v:3}) = {sigmoid(v):.4f}")

Hồi quy logistic = hồi quy tuyến tính + sigmoid. Đó cũng là cách lớp cuối của mạng neural phân loại nhị phân hoạt động.

## 8. Ngưỡng quyết định — không phải lúc nào cũng 0.5

In [ ]:
xac_suat = np.array([0.1, 0.35, 0.45, 0.55, 0.7, 0.92])
that = np.array([0, 0, 1, 0, 1, 1])

print(f"{'xác suất':>10} {'thật':>6} | " + " | ".join(f"ngưỡng {t}" for t in (0.3, 0.5, 0.7)))
print("-" * 62)
for p, t in zip(xac_suat, that):
    dd = " | ".join(f"{'   1   ' if p >= ng else '   0   '}" for ng in (0.3, 0.5, 0.7))
    print(f"{p:>10} {t:>6} | {dd}")

print()
for ng in (0.3, 0.5, 0.7):
    print(f"  ngưỡng {ng}: bắt được {((xac_suat >= ng) & (that == 1)).sum()}/3 ca dương, "
          f"báo nhầm {((xac_suat >= ng) & (that == 0)).sum()} ca")

**Ngưỡng là lựa chọn kinh doanh, không phải mặc định kỹ thuật.**

| Bài toán | Nên chọn | Vì sao |
|---|---|---|
| Sàng lọc ung thư | Ngưỡng **thấp** | Bỏ sót bệnh nhân nguy hiểm hơn báo nhầm |
| Lọc email spam | Ngưỡng **cao** | Đánh nhầm email quan trọng vào spam rất tệ |
| Duyệt vay | Tuỳ khẩu vị rủi ro | Cân giữa nợ xấu và bỏ lỡ khách tốt |

Bạn sẽ học đo hai loại sai này bằng **precision** và **recall** ở Phase 04.

## 9. Vì sao phải đặt seed

In [ ]:
print("Không seed — mỗi lần chạy một khác:")
print(" ", np.random.default_rng().normal(size=3).round(3))
print(" ", np.random.default_rng().normal(size=3).round(3))

print("\nCó seed — luôn giống hệt:")
print(" ", np.random.default_rng(42).normal(size=3).round(3))
print(" ", np.random.default_rng(42).normal(size=3).round(3))

Không có seed thì bạn không thể biết kết quả tốt hơn là do **cải tiến thật** hay do **may mắn ngẫu nhiên**.

> 📌 `np.random.seed(42)` là cách cũ (dùng biến toàn cục, dễ bị thư viện khác đụng). Cách hiện đại là `rng = np.random.default_rng(42)` rồi dùng `rng.` — mỗi chỗ một generator riêng.

## ✍️ Bài tập

Phase 02 có 2 bài tập:

```bash
pytest tests/phase02 -v
```

Xong rồi thì đọc [8 câu hỏi phỏng vấn](../../../resources/interview/phase02-math.md) và **trả lời thành tiếng**.